# Deep Research Agent with Oracle AI Agent Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/agent_memory/01_deep_research_openai_agents.ipynb) [![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)


**Framework:** OpenAI Agents SDK · **Web search:** Tavily · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook walks through building a **deep research agent** for human genome exploration. The agent uses Tavily for live web research, and stores its running conversation and durable findings in Oracle AI Database via the `oracleagentmemory` package.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational | Customer support, coding copilot, chat UIs |
| **Deep Research** | Multi-step autonomous investigation | Literature review, market research, technical scoping |
| **Workflow** | Predetermined sequence of steps, often with conditional branches | Approval pipelines, compliance checks, structured triage |

> **📌 This notebook focuses on the Deep Research mode.**
>
> A deep-research agent plans, retrieves, synthesises, and accumulates findings over long-horizon investigations. Memory is central — without durable memory, every session restarts from zero and the agent cannot build on prior research.

## What You'll Learn

- How to implement the **OpenAI Agents SDK `Session` protocol** against a custom backend (Oracle AI Agent Memory)
- How to wrap **Tavily** as a `function_tool` the agent can call
- How to store long-lived research findings as durable **memories** (separate from short-term conversation history)
- How to verify memory continuity across sessions — the agent can pick up where it left off

> **💡 Key insight:** Short-term memory (conversation history for the current run) and long-term memory (durable findings across runs) are different access patterns over the same store. We use `Session` for the former and `add_memory()` for the latter.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **`OPENAI_API_KEY`** for the agent's LLM and embeddings
- **`TAVILY_API_KEY`** for web search (free tier available at [tavily.com](https://tavily.com))
- The `oracleagentmemory` wheel installed in the active kernel's environment

## 1. Install dependencies

We need four packages: `oracleagentmemory[litellm]` (memory + embeddings), `openai-agents` (agent framework), `tavily-python` (web search), and `nest_asyncio` (so `Runner.run` works cleanly inside a Jupyter event loop).

In [1]:
%pip install -q "oracleagentmemory[litellm]" openai-agents tavily-python nest_asyncio

Note: you may need to restart the kernel to use updated packages.


## 2. Configure API keys and Oracle connection

Set environment variables for OpenAI, Tavily, and the Oracle database. We use `os.environ.setdefault` so shell or `.env` values still win if you've set them elsewhere.

In [ ]:
import os

# LLM + embedding provider (used by both the agent and OAMP)
os.environ.setdefault("OPENAI_API_KEY", "sk-")

# Tavily for web search
os.environ.setdefault("TAVILY_API_KEY", "tvly-")

# Oracle AI Database connection
os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

# Jupyter runs an async event loop already — this lets us call async SDK methods cleanly
import nest_asyncio
nest_asyncio.apply()

print("Environment configured.")

Environment configured.


## 3. Connect to Oracle AI Database and initialise the memory client

`OracleAgentMemory` is the governed memory client. With `extract_memories=True` and an `llm` attached, the client will automatically extract durable memories from messages added to a thread. We use `text-embedding-3-small` for semantic search over the memory store.

> **💡 Key insight:** `schema_policy` controls how the library interacts with the DB on startup. `REQUIRE_EXISTING` (default) never issues DDL — safe for production. `CREATE_IF_NECESSARY` fills in missing objects — safe for dev. `RECREATE` drops and rebuilds — useful when you've changed embedding dimensions.

In [3]:
import oracledb
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

# The LLM the extraction pipeline will use to turn raw messages into durable memories
extraction_llm = Llm("gpt-4o-mini", temperature=0.2)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",   # OpenAI-compatible; 1536 dims
    llm=extraction_llm,                   # enables automatic memory extraction
    extract_memories=True,
    schema_policy="recreate",
    table_name_prefix="genome_",         # isolates this example's tables from others
)
print("Memory client ready.")

Connected to Oracle AI Database: 23.26.0.0.0
Memory client ready.


## 4. Register the research user and agent

Oracle AI Agent Memory scopes every record by `user_id` and `agent_id`. Registering them up-front gives the store a stable identity to hang memories on and lets us enforce tenant-style isolation across multiple users.

> **📌 Scoping matters for production.** In a multi-user deployment, `user_id` should match your application's authenticated user. The memory client's `search()` method requires a concrete `user_id` on every call — an intentional guardrail against cross-tenant leaks.

In [4]:
USER_ID = "genome-researcher-1"
AGENT_ID = "deep-research-agent"

for create_fn, eid, info in [
    (memory_client.add_user, USER_ID, "Genomics researcher exploring human disease associations."),
    (memory_client.add_agent, AGENT_ID, "Deep-research agent with web search and long-term memory."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

Registered: genome-researcher-1
Registered: deep-research-agent


## 5. Implement the `Session` protocol on top of Oracle AI Agent Memory

The OpenAI Agents SDK defines a `Session` protocol with four async methods:

| Method | Purpose |
|---|---|
| `get_items(limit)` | Return the conversation history the runner should inject |
| `add_items(items)` | Persist new items the runner produced during a run |
| `pop_item()` | Remove and return the most recent item (for corrections) |
| `clear_session()` | Drop everything for this session |

Implementing this protocol against Oracle AI Agent Memory gives us **exact-resumption** short-term memory — the next time this session runs, the agent receives the full prior conversation without any manual wiring.

> **💡 Key insight:** The `Session` protocol handles *short-term* memory (the items the runner replays each turn). Long-term memory (durable facts the agent should remember across sessions) is a separate concern that we handle with `add_memory()` below.

In [5]:
import json
from typing import Optional
from agents.memory.session import SessionABC


class OracleAgentMemorySession(SessionABC):
    """Session backend that persists OpenAI Agents SDK items in Oracle AI Agent Memory.

    Each item is serialised to JSON and stored as a memory record tagged with the
    session's thread_id. Ordering is preserved by the store's insertion timestamp.
    """

    def __init__(self, session_id: str, client: OracleAgentMemory, user_id: str, agent_id: str):
        self.session_id = session_id
        self._client = client
        self._store = client._store
        self._user_id = user_id
        self._agent_id = agent_id
        # Ensure the underlying thread exists so memories can attach to it
        try:
            self._client.create_thread(
                thread_id=session_id, user_id=user_id, agent_id=agent_id,
            )
        except ValueError:
            pass  # thread already exists

    async def get_items(self, limit: Optional[int] = None) -> list:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=limit if limit else 10_000,
            metadata_filter={"session_item": True},
        )
        items = [json.loads(r.content) for r in records]
        return items[-limit:] if limit else items

    async def add_items(self, items: list) -> None:
        if not items:
            return
        for item in items:
            self._client.add_memory(
                json.dumps(item),
                user_id=self._user_id,
                agent_id=self._agent_id,
                thread_id=self.session_id,
                metadata={"session_item": True},
            )

    async def pop_item(self) -> Optional[dict]:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=1,
            metadata_filter={"session_item": True},
        )
        if not records:
            return None
        # Records come back newest-first when limit=1 is applied; delete and return
        last = records[-1]
        self._store.delete("memory", last.id)
        return json.loads(last.content)

    async def clear_session(self) -> None:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=10_000,
            metadata_filter={"session_item": True},
        )
        for r in records:
            self._store.delete("memory", r.id)


print("OracleAgentMemorySession implemented.")

OracleAgentMemorySession implemented.


## 6. Define the agent's tools

The agent needs three tools:

1. **`tavily_search`** — live web search for up-to-date genomics sources
2. **`save_research_finding`** — persist a durable research note the agent can recall in future sessions
3. **`recall_research_findings`** — search the long-term memory for prior findings on a topic

Tools 2 and 3 are how the agent manages its **long-term** memory. They're distinct from the `Session` machinery, which handles short-term conversation history automatically.

> **📌 The `@function_tool` decorator** auto-generates the tool's JSON schema from Python type hints and parses the docstring for the description the LLM sees. Write informative docstrings — the LLM reads them.

In [6]:
from agents import function_tool
from tavily import TavilyClient

_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """Search the live web for recent, authoritative genomics sources.

    Use this to pull current information from NCBI, OMIM, peer-reviewed journals,
    and other scientific sources. Prefer this over relying on model-internal knowledge
    for facts about specific genes, mutations, or recent studies.

    Args:
        query: Natural-language search query.
        max_results: How many results to return (1-10).
    """
    resp = _tavily.search(
        query=query, max_results=max_results,
        search_depth="advanced", include_answer=True,
    )
    lines = [f"Answer: {resp.get('answer', '')}"]
    for r in resp.get("results", []):
        lines.append(f"- {r['title']} ({r['url']})\n  {r['content'][:300]}")
    return "\n".join(lines)


@function_tool
def save_research_finding(topic: str, finding: str) -> str:
    """Persist a durable research finding the agent should recall in future sessions.

    Use this for claims worth remembering across sessions — e.g. "BRCA1 mutations
    are associated with 45-85% lifetime breast cancer risk in women." Do not use
    this for conversational acknowledgements or speculation.

    Args:
        topic: Short topic tag (e.g. 'BRCA1', 'TP53 mutations').
        finding: The finding to remember, ideally one sentence.
    """
    memory_id = memory_client.add_memory(
        f"[{topic}] {finding}",
        user_id=USER_ID,
        agent_id=AGENT_ID,
        metadata={"topic": topic, "kind": "research_finding"},
    )
    return f"Saved finding (id={memory_id})."


@function_tool
def recall_research_findings(query: str, max_results: int = 5) -> str:
    """Search prior research findings for information relevant to a query.

    Use this at the start of a research task to check what is already known.
    Results are ranked by semantic similarity.

    Args:
        query: Natural-language query describing what you want to recall.
        max_results: How many findings to return.
    """
    results = memory_client.search(
        query, user_id=USER_ID, agent_id=AGENT_ID, max_results=max_results,
    )
    if not results:
        return "(no prior findings)"
    return "\n".join(
        f"- {r.content}  [distance={r.distance:.3f}]" for r in results
    )


print("Tools defined: tavily_search, save_research_finding, recall_research_findings")

Tools defined: tavily_search, save_research_finding, recall_research_findings


## 7. Construct the research agent

The agent's `instructions` are its system prompt — the place to encode the behaviour you want. For a deep-research agent, the instructions should emphasise:

1. **Recall before research** — check long-term memory before hitting the web
2. **Cite sources** — so findings are traceable
3. **Save what's worth remembering** — commit durable conclusions explicitly

> **💡 Key insight:** A deep-research agent's output quality is largely determined by how well its instructions distinguish *research* (retrieve + synthesise) from *conversation* (respond + acknowledge). Encode that distinction in the system prompt.

In [7]:
from agents import Agent, Runner

INSTRUCTIONS = """You are a deep-research agent specialising in human genome exploration.

For every research question:
1. FIRST call `recall_research_findings` to check what is already known from prior sessions.
2. If the prior findings are insufficient or outdated, call `tavily_search` for current sources.
3. Synthesise a clear, cited answer. Prefer authoritative sources (NCBI, OMIM, PubMed).
4. Call `save_research_finding` for each durable conclusion worth remembering across sessions.
   Save one finding per call; keep findings atomic and one sentence long.
5. Present the final answer to the user with inline citations (URLs).

Do not save conversational acknowledgements as findings. Only save factual conclusions.
"""

research_agent = Agent(
    name="GenomeDeepResearcher",
    instructions=INSTRUCTIONS,
    tools=[tavily_search, save_research_finding, recall_research_findings],
    model="gpt-5.4",
)
print(f"Agent created: {research_agent.name}")

Agent created: GenomeDeepResearcher


## 8. Run a research session

We create a session (tied to a `thread_id` in the memory store) and run the agent over a sequence of research questions. Because we pass `session=session`, the SDK will automatically inject prior conversation items at the start of each turn and persist new items at the end.

> **📌 Separation of concerns.**
> - The `Session` persists the **raw conversation** (working memory).
> - The agent's `save_research_finding` tool persists **durable findings** (long-term memory).
> Both live in the same Oracle database but are accessed through different interfaces.

In [8]:
SESSION_ID = "genome-session-001"
session = OracleAgentMemorySession(
    session_id=SESSION_ID, 
    client=memory_client,
    user_id=USER_ID, 
    agent_id=AGENT_ID,
)

research_questions = [
    "What is BRCA1 and what is its primary function in DNA repair?",
    "What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?",
    "How does BRCA1 interact with BRCA2 in homologous recombination?",
]

for i, q in enumerate(research_questions, 1):
    print(f"\n{'=' * 70}\nQ{i}: {q}\n{'=' * 70}")
    result = await Runner.run(research_agent, q, session=session)
    print(result.final_output)


Q1: What is BRCA1 and what is its primary function in DNA repair?
BRCA1 is a human **tumor suppressor gene** that encodes a nuclear phosphoprotein involved in maintaining **genomic stability**. It works with other proteins in the cellular response to DNA damage and is best known for its role in hereditary breast and ovarian cancer when mutated ([NCBI Gene](https://www.ncbi.nlm.nih.gov/datasets/gene/672/), [MedlinePlus Genetics](https://medlineplus.gov/genetics/gene/brca1/)).

Its **primary function in DNA repair** is to help repair **DNA double-strand breaks** through **homologous recombination**, a high-fidelity repair pathway that uses an intact sister chromatid as a template. In short, BRCA1 helps cells repair dangerous DNA breaks accurately rather than leaving them to be repaired by more error-prone mechanisms ([NCBI Gene](https://www.ncbi.nlm.nih.gov/datasets/gene/672/), [MedlinePlus Genetics](https://medlineplus.gov/genetics/gene/brca1/), [PMC review](https://pmc.ncbi.nlm.nih.go

## 9. Inspect what the agent remembered

At this point the agent has accumulated both short-term conversation items (via the session) and long-term research findings (via `save_research_finding`). Let's inspect both.

In [9]:
# Short-term: conversation items replayed each turn
session_items = await session.get_items()
print(f"Session items: {len(session_items)}")
for it in session_items[:3]:
    print(f"  - {str(it)[:120]}...")

# Long-term: durable findings the agent chose to save
findings = memory_client.search(
    "BRCA1 function and risk", user_id=USER_ID, agent_id=AGENT_ID, max_results=10,
)
print(f"\nDurable research findings: {len(findings)}")
for r in findings:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

Session items: 28
  - {'content': 'What is BRCA1 and what is its primary function in DNA repair?', 'role': 'user'}...
  - {'arguments': '{"query":"BRCA1 gene primary function in DNA repair", "max_results": 5}', 'call_id': 'call_h7T59GMFF87r0g...
  - {'call_id': 'call_h7T59GMFF87r0g2JshWq81nF', 'output': '- {"content": "What is BRCA1 and what is its primary function in...


/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/oracleagentmemory/core/oracleagentmemory.py:698: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 
  return run_async_in_sync(self._do_search_async, query, scope, max_results, record_types)



Durable research findings: 10
  - [BRCA1] BRCA1 encodes a tumor-suppressor protein that helps maintain genomic stability by coordinating the cellular response to DNA damage.  [distance=0.275]
  - {"call_id": "call_hpWi8B9FcRzWOF8FZiFcapAh", "output": "Answer: The BRCA1 gene's primary function is DNA repair via homologous recombination, which helps maintain genomic stability and prevent tumor development. Mutations in BRCA1 lead to impaired DNA repair, increasing cancer risk. BRCA1 also plays a role in transcription regulation and cell signaling.\n- What is the Role of BRCA1 in Normal Cells? (https://www.news-medical.net/health/What-is-the-Role-of-BRCA1-in-Normal-Cells.aspx)\n  ## DNA repair and recombination\n\nThe BRCA1 protein is an E3 uniquitin-protein ligase that regulates the creation of Lys-6-linked polyubiquitin chains and makes an important contribution toward DNA repair by enabling cells to respond to DNA damage.\n\nThis maintenance of the DNA is a crucial part of BR\n- BRCA1

## 10. Verify continuity — resume a fresh session with the same memory store

The real test of a memory-aware agent is whether it can pick up where a prior session left off. Let's create a *new* session (simulating a separate process or later day) and ask a question that builds on prior findings.

If the agent recalls BRCA1 findings from the previous run without re-searching, we've demonstrated end-to-end memory continuity.

In [10]:
# Simulate a fresh session (new session_id, but same user/agent)
FRESH_SESSION_ID = "genome-session-002"
fresh_session = OracleAgentMemorySession(
    session_id=FRESH_SESSION_ID, client=memory_client,
    user_id=USER_ID, agent_id=AGENT_ID,
)

followup = "Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening."

print(f"FRESH SESSION — Q: {followup}\n")
result = await Runner.run(research_agent, followup, session=fresh_session)
print(result.final_output)

FRESH SESSION — Q: Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening.

A patient with a **pathogenic BRCA1 variant** would usually be counselled that screening needs to start **earlier and be more intensive** than for the general population, because BRCA1 is associated with a markedly increased lifetime risk of **breast cancer** and a substantial risk of **ovarian cancer** ([NCI BRCA fact sheet](https://www.cancer.gov/about-cancer/causes-prevention/genetics/brca-fact-sheet), [NCI PDQ](https://www.cancer.gov/publications/pdq/information-summaries/genetics/brca-genes-hp-pdq)).

### Breast screening
Counselling would typically include:

- **Annual breast MRI starting around age 25**
- **Annual mammography added at age 30**
- Ongoing **clinical breast awareness** and prompt assessment of any new breast symptoms

This approach is used because MRI is more sensitive in younger high-risk women, and current high

## 11. Cleanup (optional)

Uncomment the cell below to clear all session items and findings for this example. Leave it commented to keep the data for subsequent runs — that's often the point.

In [11]:
# await session.clear_session()
# await fresh_session.clear_session()
# for r in memory_client.search("BRCA", user_id=USER_ID, agent_id=AGENT_ID, max_results=100):
#     memory_client.delete_memory(r.record.id)
# print("Cleaned up.")

connection.close()
print("Connection closed.")

Connection closed.


## Key Takeaways

> **🎯 1. The `Session` protocol is the clean integration point.** Four async methods (`get_items`, `add_items`, `pop_item`, `clear_session`) is all the OpenAI Agents SDK needs to plug a custom memory backend in. You don't have to modify the runner — you implement the protocol and pass an instance via `session=`.

> **🎯 2. Separate short-term and long-term memory concerns.** The session handles conversation replay for the current agent loop. Durable findings (the ones worth recalling in a future session) should be written through tools the agent calls deliberately, not dumped into the conversation log.

> **🎯 3. Instructions steer memory usage.** A deep-research agent must be explicitly told to *recall before researching* and *save durable conclusions*. Otherwise the LLM will treat memory as optional decoration and the store will fill up with nothing useful.

> **🎯 4. Oracle AI Database gives you one substrate for both tiers.** Short-term items, long-term findings, and any structured business data the agent needs to reference all live in one governed backend — one pool, one backup, one compliance review.

> **🎯 5. Continuity is testable.** A new `session_id` with the same `user_id` / `agent_id` should produce an agent that reasons over prior long-term findings. That's the simplest end-to-end test of a memory-aware agent.